Original model drawn from https://simtk.org/projects/rat_hlimb_model which is based on Johnson et al. 2008 [@missing-reference].

In [1]:
import opensim as osim
from pathlib import Path
import polars as pl
import numpy as np
import re

model_dir = Path('./models/osim')
model_file = model_dir / 'rat_hindlimb.osim'

model = osim.Model(model_file.as_posix())
muscles = model.getMuscles()

with open(model_file, 'r') as file:
    file_content = file.read()

terms = [
    'pelvis', 'femur', 'tibia', 'foot',        # Bodies
    'sacroiliac', 'hip', 'knee', 'ankle'       # Joints
]

for term in terms:
    # Pattern explanation: Find 'term', but ONLY if not followed by '_r'
    file_content = re.sub(rf'{term}(?!_r)', f'{term}_r', file_content)

file_content = file_content.replace('.vtp', '.stl')

# Update muscle names
for muscle in muscles:
    muscle_name = muscle.getName()
    if not muscle_name.startswith('R_'):
      file_content = file_content.replace(muscle_name, 'R_' + muscle_name )

with open(model_file, 'w') as file:
    file.write(file_content)

[info] Updating Model file from 40000 to latest format...
[info] Loaded model RatHindlimbRight from file models/osim/rat_hindlimb.osim


## Conversion to Millard muscles
The original model uses Thelen [@missing-reference] muscle models, which have been largely superseded by Millard [@missing-reference] muscle models in OpenSim. We can convert the muscles using the [`model_thelen_to_millard`](src/rathindlimb/muscle_utils.py) function. 

In [2]:
from src.rathindlimb import model_thelen_to_millard
model = osim.Model(model_file.as_posix())
millard_model = model_thelen_to_millard(model)

out = model_dir / 'rat_hindlimb_millard.osim'
millard_model.printToXML(out.as_posix())

[info] Updating Model file from 40000 to latest format...


True

[info] Loaded model RatHindlimbRight from file models/osim/rat_hindlimb.osim


## Lock unused coordinates
We lock sacroiliac flexion, ankle internal rotation and ankle adduction as we do not have the resolution to capture the small amount of motion that these correspond to. 

In [3]:
from movedb.osim import OsimGraph
coords_to_lock = ["ankle_r_add", "ankle_r_int", "sacroiliac_r_flx"]
joints = millard_model.getJointSet()
millard_model.initSystem()
graph = OsimGraph(osim_model=millard_model)

for coord_name in coords_to_lock:
  coord = graph.get_coordinate(coord_name)
  coord.set_locked(True)

out = model_dir / 'rat_hindlimb_millard_locked.osim'
graph.osim_model.finalizeFromProperties()
graph.osim_model.finalizeConnections()
graph.osim_model.printToXML(out.as_posix())

2025-12-10 05:38:11.476 | DEBUG    | movedb.osim.analysis.osim_graph:create_rigid_graph:179 - Coordinate sacrum_pitch range: (-1.57079633, 1.57079633)


2025-12-10 05:38:11.477 | DEBUG    | movedb.osim.analysis.osim_graph:create_rigid_graph:179 - Coordinate sacrum_roll range: (-1.57079633, 1.57079633)


2025-12-10 05:38:11.477 | DEBUG    | movedb.osim.analysis.osim_graph:create_rigid_graph:179 - Coordinate sacrum_yaw range: (-1.57079633, 1.57079633)


2025-12-10 05:38:11.477 | DEBUG    | movedb.osim.analysis.osim_graph:create_rigid_graph:179 - Coordinate sacrum_x range: (-0.1, 0.25)


2025-12-10 05:38:11.477 | DEBUG    | movedb.osim.analysis.osim_graph:create_rigid_graph:179 - Coordinate sacrum_y range: (0.0, 0.2)


...

2025-12-10 05:38:11.509 | DEBUG    | movedb.osim.analysis.osim_graph:cache_muscle_crossings:307 - Muscle R_FHL crosses joints {'ankle_r'}


2025-12-10 05:38:11.509 | DEBUG    | movedb.osim.analysis.osim_graph:cache_muscle_crossings:307 - Muscle R_Per crosses joints {'ankle_r'}


2025-12-10 05:38:11.510 | DEBUG    | movedb.osim.analysis.osim_graph:cache_muscle_crossings:307 - Muscle R_Pop crosses joints {'knee_r'}


[info] 'R_BFa' Parameter update for the damped-model: ActiveForceLengthCurve minimum value was 0.1 but is now 0.0.
[info] 'R_BFp' Parameter update for the damped-model: ActiveForceLengthCurve minimum value was 0.1 but is now 0.0.
[info] 'R_CF' Parameter update for the damped-model: ActiveForceLengthCurve minimum value was 0.1 but is now 0.0.
[info] 'R_STp' Parameter update for the damped-model: ActiveForceLengthCurve minimum value was 0.1 but is now 0.0.
[info] 'R_STa' Parameter update for the damped-model: ActiveForceLengthCurve minimum value was 0.1 but is now 0.0.
[info] 'R_SM' Parameter update for the damped-model: ActiveForceLengthCurve minimum value was 0.1 but is now 0.0.
[info] 'R_QF' Parameter update for the damped-model: ActiveForceLengthCurve minimum value was 0.1 but is now 0.0.
[info] 'R_TFL' Parameter update for the damped-model: ActiveForceLengthCurve minimum value was 0.1 but is now 0.0.
[info] 'R_GMa' Parameter update for the damped-model: ActiveForceLengthCurve minimu

True

## Muscle path analysis
The model is moved through its full range of motion and muscle lengths are analyzed to identify discontinuties that may indicate problems with muscle path definitions. The `OsimGraph` class from `movedb` is used to represent an OpenSim model as a graph structure, which allows for efficient analysis of muscle paths.

In [4]:
model = osim.Model(out.as_posix())
graph = OsimGraph(osim_model=model)
results = graph.get_all_muscle_lengths_rom(min_points=10000)

2025-12-10 05:38:11.565 | DEBUG    | movedb.osim.analysis.osim_graph:create_rigid_graph:179 - Coordinate sacrum_pitch range: (-1.57079633, 1.57079633)


[info] Loaded model RatHindlimbRight from file models/osim/rat_hindlimb_millard_locked.osim


2025-12-10 05:38:11.565 | DEBUG    | movedb.osim.analysis.osim_graph:create_rigid_graph:179 - Coordinate sacrum_roll range: (-1.57079633, 1.57079633)


2025-12-10 05:38:11.566 | DEBUG    | movedb.osim.analysis.osim_graph:create_rigid_graph:179 - Coordinate sacrum_yaw range: (-1.57079633, 1.57079633)


2025-12-10 05:38:11.566 | DEBUG    | movedb.osim.analysis.osim_graph:create_rigid_graph:179 - Coordinate sacrum_x range: (-0.1, 0.25)


...

[warning] Coordinate.setValue:  coordinate sacroiliac_r_flx is locked. Unable to change its value.


2025-12-10 05:38:20.026 | WARNING  | movedb.osim.analysis.osim_graph:get_all_muscle_lengths_rom:421 - Locked coordinates frozenset({'ankle_r_add', 'ankle_r_int'}) for muscles {'R_LG', 'R_MG', 'R_Pla'}


[warning] Coordinate.setValue:  coordinate ankle_r_int is locked. Unable to change its value.
[warning] Coordinate.setValue:  coordinate ankle_r_add is locked. Unable to change its value.


2025-12-10 05:38:20.535 | WARNING  | movedb.osim.analysis.osim_graph:get_all_muscle_lengths_rom:421 - Locked coordinates frozenset({'ankle_r_add', 'ankle_r_int'}) for muscles {'R_Sol', 'R_EDL', 'R_FDL', 'R_TA', 'R_TP', 'R_Per', 'R_FHL'}


[warning] Coordinate.setValue:  coordinate ankle_r_int is locked. Unable to change its value.
[warning] Coordinate.setValue:  coordinate ankle_r_add is locked. Unable to change its value.


The muscle lengths are plotted on parallel coordinate plots with the joint angles that influence them. This allows for visual identification of muscles with discontinuous length changes.

In [5]:
#| eval: false
import plotly.graph_objects as go
figure_dir = Path('./figs')

for muscle_name, muscle_data in results.items():
  fig = go.Figure(
    data=go.Parcoords(
      line=dict(color=muscle_data[muscle_name], colorscale='Viridis', showscale=True),
      dimensions=[
        dict(label=col, values=muscle_data[col]) for col in muscle_data.columns
      ]
    )
  )
  fig.update_layout(title=f'{muscle_name} Lengths')
  # Write to png
  fig_path = figure_dir / muscle_name 
  fig_path.mkdir(exist_ok=True)
  fig.write_image(fig_path / f'{muscle_name}_orig_lengths.png')

  # TODO: Should this also include moment arms?

### Discontinuous muscles
The following muscles were identified as having discontinuous length changes:

- BFa 
![BFa Length](./figs/BFa_orig_length.png)
- CF
![CF Length](./figs/CF_orig_length.png)
- STp
![STp Length](./figs/STp_orig_length.png)
- TA
![TA Length](./figs/TA_orig_length.png)
- VL
![VL Length](./figs/VL_orig_length.png)

Investigating the root cause of this, we found that the likely culprit is a hard-coded value in [OpenSim's WrapCylinder.cpp code](https://github.com/opensim-org/opensim-core/blob/f9cd558ec3ea99dda206e5e412e62e23cf19bd7e/OpenSim/Simulation/Wrap/WrapCylinder.cpp#L668) that determines the number of segments used to approximate the muscle path over a wrapping cylinder:

```c
// Each muscle segment on the surface of the cylinder should be
// 0.002 meters long. This assumes the model is in meters, of course.
int numWrapSegments = (int)(aWrapResult.wrap_path_length / 0.002);
if (numWrapSegments < 1) numWrapSegments = 1;
```

This means that for cylinders with small circumferences, the muscle path may be approximated with very few segments, leading to abrupt changes in path length as the muscle wraps around the cylinder. To address this issue, we either need to modify the OpenSim source code to increase the number of segments used for wrapping or modify the wrapping paths in the model. 

### Muscle path redefinition
Johnson's original model relied heavily on wrapping surfaces, but a more recent work that built upon that original model and drew from Greene's original anatomical text defined a large set of attachment points for muscles instead of relying on wrapping surfaces. This newer model may avoid some of the issues with discontinuous muscle lengths. Unfortunately, that model was created for a software called [Animatlab](https://www.animatlab.com/), and there was no existing OpenSim version of it. 

We were able to get in touch with the author of the Animatlab model, Dr. Young, who provided the original files and assisted in understanding the model structure. From there, we manually recorded the muscle attachment points seen in the model as seen in @tbl-young-attachments.

In [6]:
#| label: tbl-young-attachments
from IPython.display import Markdown as md
from tabulate import tabulate

data = pl.read_csv('./data/attachments/young_model_attachments.csv')

md(tabulate(data, headers='keys', tablefmt='pipe'))

| 0      | 1         | 2      | 3         | 4      | 5         | 6      | 7      | 8      | 9         | 10     | 11      | 12        | 13     | 14     | 15     | 16        | 17     | 18        | 19      | 20        | 21     | 22        | 23      | 24        | 25      | 26        | 27     | 28        | 29     | 30        | 31      | 32        | 33     | 34    | 35      | 36     | 37        | 38     | 39     | 40      | 41     | 42        | 43     | 44      | 45       | 46     | 47     | 48        | 49     | 50      | 51     | 52     | 53     | 54        | 55     | 56     | 57      | 58     | 59        | 60     | 61      | 62     | 63     | 64     | 65        | 66     | 67     | 68      | 69     | 70        | 71     | 72      | 73     | 74        | 75     | 76      | 77      | 78     | 79        | 80     | 81     | 82     | 83     | 84        | 85     | 86     | 87     | 88        | 89     | 90        | 91     | 92     | 93     | 94        | 95     | 96        | 97     | 98      | 99     | 100               | 101    | 102     | 103    | 104       | 105    | 106      | 107    | 108       | 109    | 110       | 111    | 112    | 113    | 114       | 115     | 116       | 117    | 118       | 119    | 120    | 121    | 122    | 123       |
|:-------|:----------|:-------|:----------|:-------|:----------|:-------|:-------|:-------|:----------|:-------|:--------|:----------|:-------|:-------|:-------|:----------|:-------|:----------|:--------|:----------|:-------|:----------|:--------|:----------|:--------|:----------|:-------|:----------|:-------|:----------|:--------|:----------|:-------|:------|:--------|:-------|:----------|:-------|:-------|:--------|:-------|:----------|:-------|:--------|:---------|:-------|:-------|:----------|:-------|:--------|:-------|:-------|:-------|:----------|:-------|:-------|:--------|:-------|:----------|:-------|:--------|:-------|:-------|:-------|:----------|:-------|:-------|:--------|:-------|:----------|:-------|:--------|:-------|:----------|:-------|:--------|:--------|:-------|:----------|:-------|:-------|:-------|:-------|:----------|:-------|:-------|:-------|:----------|:-------|:----------|:-------|:-------|:-------|:----------|:-------|:----------|:-------|:--------|:-------|:------------------|:-------|:--------|:-------|:----------|:-------|:---------|:-------|:----------|:-------|:----------|:-------|:-------|:-------|:----------|:--------|:----------|:-------|:----------|:-------|:-------|:-------|:-------|:----------|
| AB     | AB        | AL     | AL        | AM     | AM        | BFa    | BFa    | BFa    | BFa       | BFp    | BFp     | BFp       | CF     | CF     | CF     | CF        | GI     | GI        | GS      | GS        | GMa    | GMa       | GMe     | GMe       | GMi     | GMi       | GA     | GA        | GP     | GP        | IP      | IP        | EDL    | EDL   | EDL     | EDL    | EDL       | LG     | LG     | LG      | LG     | LG        | FDL    | FDL     | FDL      | FDL    | FDL    | FDL       | FHL    | FHL     | FHL    | FHL    | FHL    | FHL       | MG     | MG     | MG      | MG     | MG        | Per    | Per     | Per    | Per    | Per    | Per       | Pla    | Pla    | Pla     | Pla    | Pla       | Sol    | Sol     | Sol    | Sol       | TA     | TA      | TA      | TA     | TA        | TP     | TP     | TP     | TP     | TP        | Pop    | Pop    | Pop    | Pop       | SM     | SM        | STa    | STa    | STa    | STa       | STp    | STp       | VI     | VI      | VI     | VI                | VL     | VL      | VL     | VL        | VM     | VM       | VM     | VM        | OE     | OE        | OI     | OI     | Pec    | Pec       | Pir     | Pir       | QF     | QF        | RF     | RF     | RF     | TFL    | TFL       |
| Origin | Insertion | Origin | Insertion | Origin | Insertion | Origin | Via    | Via    | Insertion | Origin | Via     | Insertion | Origin | Via    | Via    | Insertion | Origin | Insertion | Origin  | Insertion | Origin | Insertion | Origin  | Insertion | Origin  | Insertion | Origin | Insertion | Origin | Insertion | Origin  | Insertion | Origin | Via   | Via     | Via    | Insertion | Origin | Via    | Via     | Via    | Insertion | Origin | Via     | Via      | Via    | Via    | Insertion | Origin | Via     | Via    | Via    | Via    | Insertion | Origin | Via    | Via     | Via    | Insertion | Origin | Via     | Via    | Via    | Via    | Insertion | Origin | Via    | Via     | Via    | Insertion | Origin | Via     | Via    | Insertion | Origin | Via     | Via     | Via    | Insertion | Origin | Via    | Via    | Via    | Insertion | Origin | Via    | Via    | Insertion | Origin | Insertion | Origin | Via    | Via    | Insertion | Origin | Insertion | Origin | Via     | Via    | Insertion         | Origin | Via     | Via    | Insertion | Origin | Via      | Via    | Insertion | Origin | Insertion | Origin | Via    | Origin | Insertion | Origin  | Insertion | Origin | Insertion | Origin | Via    | Via    | Origin | Insertion |
| Pelvis | Femur     | Pelvis | Femur     | Pelvis | Tibia     | Pelvis | Pelvis | Pelvis | Tibia     | Pelvis | Tibia   | Tibia     | Pelvis | Pelvis | Pelvis | Femur     | Pelvis | Femur     | Pelvis  | Femur     | Pelvis | Femur     | Pelvis  | Femur     | Pelvis  | Femur     | Pelvis | Tibia     | Pelvis | Tibia     | Pelvis  | Femur     | Femur  | Tibia | Tibia   | Foot   | Foot      | Femur  | TIbia  | Tibia   | Foot   | Foot      | Tibia  | Tibia   | Foot     | Foot   | Foot   | Foot      | Tibia  | Tibia   | Foot   | Foot   | Foot   | Foot      | Femur  | Tibia  | Tibia   | Foot   | Foot      | Tibia  | Tibia   | Foot   | Foot   | Foot   | Foot      | Femur  | Tibia  | Tibia   | Foot   | Foot      | Tibia  | Tibia   | Foot   | Foot      | Tibia  | Tibia   | Tibia   | Foot   | Foot      | Tibia  | Tibia  | Foot   | Foot   | Foot      | Femur  | Femur  | Tibia  | Tibia     | Pelvis | Tibia     | Pelvis | Pelvis | Pelvis | Tibia     | Pelvis | Tibia     | Femur  | Femur   | Tibia  | Tibia             | Femur  | Femur   | Tibia  | Tibia     | Femur  | Femur    | Tibia  | Tibia     | Pelvis | Femur     | Pelvis | Femur  | Pelvis | Femur     | Pelvis  | Femur     | Pelvis | Femur     | Pelvis | Femur  | Tibia  | Pelvis | Femur     |
| 17.501 | -7.993    | 10.522 | -6.94     | 17.954 | -3.0      | 19.501 | 19.878 | 19.844 | 1.844     | 22.456 | 0.864   | 0.048477  | 14.103 | 13.65  | 12.48  | -8.599    | 18.827 | -2.764    | -12.003 | -4.996    | -7.391 | -9.394    | -18.604 | -7.0      | -16.508 | -6.595    | 19.282 | -3.165    | 20.52  | -3.246    | -14.922 | -3.603    | -12.6  | 3.324 | 0.622   | 4.347  | 20.581    | -12.9  | 3.517  | 0.963   | -4.147 | -3.09     | -0.7   | -1.135  | -4.207   | -1.484 | 2.079  | 19.248    | -1.427 | -1.952  | -3.882 | -1.153 | 2.963  | 19.135    | -5.6   | -4.896 | -2.257  | -3.545 | -3.158    | 3.491  | 1.375   | -3.636 | -0.692 | 2.433  | 6.056     | -12.0  | 2.712  | 0.508   | -4.157 | -3.561    | 0.824  | -0.175  | -4.129 | -3.561    | -2.308 | -2.12   | -1.431  | 4.773  | 7.945     | -1.427 | -1.554 | -4.358 | -1.159 | 4.296     | -13.0  | -12.74 | -0.151 | -2.611    | 21.909 | -2.89     | 20.964 | 21.042 | 21.056 | -2.449    | 22.152 | -2.449    | -9.016 | -12.538 | -0.29  | -0.527            | -8.878 | -12.657 | 1.159  | -0.527    | -4.0   | -7.386   | -3.916 | -0.527    | 15.777 | -2.809    | 19.884 | -4.066 | 20.879 | -7.5      | -5.732  | -6.145    | 20.123 | -6.201    | 2.081  | -9.664 | -1.716 | -9.594 | -9.139    |
| -3.087 | -3.764    | -5.42  | -3.6      | -2.628 | -8.0      | -0.571 | -6.071 | -8.547 | -8.961    | -5.732 | -15.061 | -13.338   | -1.194 | -5.582 | -8.318 | -4.649    | -4.85  | -2.573    | -10.383 | -0.965    | -9.731 | -2.368    | -14.0   | 0.931     | -11.5   | 0.979     | -2.5   | -9.661    | -4.773 | -12.196   | -10.319 | -1.128    | -4.181 | -2.75 | -39.443 | -0.954 | -4.657    | -3.824 | -6.121 | -41.618 | -1.234 | -2.356    | -6.64  | -41.666 | -1.497   | -2.881 | -4.229 | -6.962    | -8.55  | -41.859 | -1.681 | -3.211 | -4.29  | -7.09     | -2.026 | -6.148 | -42.328 | -1.398 | -2.639    | -5.378 | -41.618 | -1.164 | -2.568 | -3.706 | -4.665    | -3.766 | -7.046 | -41.618 | -1.334 | -2.715    | -7.027 | -41.618 | -1.414 | -2.754    | -4.47  | -14.635 | -39.204 | -1.062 | -2.944    | -8.55  | -42.19 | -1.609 | -3.222 | -4.048    | -3.331 | -5.68  | -6.323 | -15.257   | -6.0   | -6.16     | -1.477 | -4.671 | -6.303 | -13.114   | -5.93  | -13.114   | -1.543 | -0.655  | 0.756  | -3.598            | -2.021 | -2.802  | 1.022  | -3.598    | -1.169 | 0.036067 | 0.713  | -3.598    | -6.5   | -2.8      | -6.8   | -1.597 | -3.121 | -4.5      | -10.422 | 1.521     | -6.0   | -3.976    | -12.0  | 0.916  | 0.815  | -9.968 | -0.491    |
| 12.02  | 17        | 8.854  | 15.053    | 13.281 | -1.151    | -5.385 | -2.285 | 1.219  | 2.876     | -0.362 | 2.014   | -3.851    | -6.141 | -2.7   | 0.92   | 28.716    | -2.4   | -0.624    | -1.711  | 5.543     | -2.277 | 7.021     | -4.228  | -2.872    | -1.771  | -1.597    | 13.354 | -2.262    | 8.583  | -3.335    | 1.129   | 3.751     | 31.579 | 1.114 | -9.139  | 0.908  | 1.023     | 29.6   | 3.926  | -6.197  | 1.309  | 1.447     | 2.559  | -6.5    | -0.06603 | 0.159  | -0.263 | 1.465     | 0.486  | -6.38   | -0.496 | -0.656 | -0.567 | -1.929    | 32.495 | 1.111  | -6.557  | -0.546 | 0.201     | 2.735  | -6.197  | 1.761  | 2.037  | 2.352  | 0.846     | 28.916 | 3.975  | -6.197  | 0.796  | 0.819     | 4.31   | -6.197  | 0.39   | 0.4       | -3.782 | -7.924  | -9.245  | -1.312 | -1.965    | 1.871  | -6.45  | -0.337 | -0.103 | -1.687    | 30.766 | 29.888 | 3.876  | -3.901    | 3.732  | -0.318    | -4.803 | -2.5   | -0.809 | -3.591    | 0.642  | -3.591    | 8.885  | 33.545  | -2.082 | -5                | 1.648  | 32.316  | -1.799 | -5        | 4.662  | 33.844   | -2.615 | -5        | 4.619  | 0.605     | 3.25   | -1.22  | 12.704 | 9.344     | -3.162  | -3.3      | 6      | -0.436    | 1      | 33.229 | -2.518 | -0.533 | 20.484    |
|        |           |        |           |        |           |        |        |        |           |        |         |           |        |        |        |           |        |           |         |           |        |           |         |           |         |           |        |           |        |           |         |           |        |       |         |        |           |        |        |         |        |           |        |         |          |        |        |           |        |         |        |        |        |           |        |        |         |        |           | PerL   |         |        |        |        | PerL      |        |        |         |        |           |        |         |        |           |        |         |         |        |           |        |        |        |        |           |        |        |        |           |        |           |        |        |        | ST?       |        |           |        |         |        | Tibial tuberosity |        |         |        |           |        |          |        |           |        |           |        |        |        |           |         |           |        |           |        |        |        |        |           |

The coordinate systems used in Animatlab differ from those in OpenSim, so a direct conversion is non-trivial. To be able to transfer the attachment points into our OpenSim model we needed to determine the necessary transformations between the coordinate systems. 
The method we chose to do this was using mesh registration to align the bone geometries from the Animatlab model to those in our OpenSim model. Once aligned, we could apply the same transformations to the muscle attachment points to bring them into the OpenSim coordinate system. 

We utilize the [`open3d`](http://www.open3d.org/) library to perform the mesh registration based on feature computation and the iterative closest point (ICP) algorithm [@missing-reference]. Additionally, we implement a function to compute the transformation matrices that will be used to transform the attachment points between models. 

TODO: Before and after and show muscle length plots to prove that what we did worked

In [7]:
from src.rathindlimb.registration import register_meshes, apply_transformation_to_mesh

# The foot mesh used in the Animatlab model is missing the phalanges, but there is a mesh that includes them that lines up with the one included in the model
mesh_dir = Path('./meshes')

source_file = mesh_dir / 'foot3_r_young_no_phalanges.stl'
target_file = mesh_dir / 'foot2_r_young.stl'
output_file1 = mesh_dir / 'foot_r_young_no_phalanges.stl' 

foot_file = mesh_dir / 'foot3_r_young.stl'
final_file = mesh_dir / 'foot_r_young.stl' 
transform = register_meshes(source_file, target_file, output_file1, seed=123)

apply_transformation_to_mesh(foot_file, transform, final_file)

Jupyter environment detected. Enabling Open3D WebVisualizer.
[Open3D INFO] WebRTC GUI backend enabled.
[Open3D INFO] WebRTCWindowSystem: HTTP handshake server disabled.


Loading and preparing meshes...
Preprocessing point clouds and computing features...


Performing global registration (RANSAC)...


[Open3D WARNING] Too few correspondences (130) after mutual filter, fall back to original correspondences.


Refining registration (ICP)...

Registered mesh saved to meshes/foot_r_young_no_phalanges.stl
Transformed mesh saved to meshes/foot_r_young.stl


TriangleMesh with 73944 points and 24648 triangles.

Then each transform is calculated for use in transforming the attachment points.

In [8]:
# Define the source, target, and output files
source_postfix = '_young.stl'
target_postfix = '_johnson.stl'
output_postfix = '.stl'
output_dir = model_dir / 'Geometry'

meshes = ['spine', 'pelvis_r', 'femur_r', 'tibia_r', 'foot_r']
transforms = {}
for mesh in meshes:
    source_file = mesh_dir / (mesh + source_postfix)
    target_file = mesh_dir / (mesh + target_postfix)
    output_file = output_dir / (mesh + output_postfix)
    transforms[mesh] = register_meshes(source_file, target_file, output_file, seed=120)

Loading and preparing meshes...


Preprocessing point clouds and computing features...


Performing global registration (RANSAC)...


Refining registration (ICP)...

Registered mesh saved to models/osim/Geometry/spine.stl
Loading and preparing meshes...


Preprocessing point clouds and computing features...


...

[Open3D WARNING] Too few correspondences (43) after mutual filter, fall back to original correspondences.


Refining registration (ICP)...

Registered mesh saved to models/osim/Geometry/tibia_r.stl
Loading and preparing meshes...


Preprocessing point clouds and computing features...


Performing global registration (RANSAC)...


Refining registration (ICP)...

Registered mesh saved to models/osim/Geometry/foot_r.stl


In [9]:
from src.rathindlimb.registration import convert_points_between_meshes
from loguru import logger

attachment_dir = Path('./data/attachments')
young_attachments = pl.read_csv(attachment_dir / 'young_model_attachments.csv')
young_attachments = young_attachments.with_columns([
    pl.col('X (mm)').cast(pl.String).str.strip_chars().cast(pl.Float64),
    pl.col('Y (mm)').cast(pl.String).str.strip_chars().cast(pl.Float64),
    pl.col('Z (mm)').cast(pl.String).str.strip_chars().cast(pl.Float64),
])
# Remove all wrap objects from the bodies
body_set: osim.BodySet = model.getBodySet()
n_bodies = body_set.getSize()
for i in range(n_bodies):
    body = model.getBodySet().get(i)
    # Remove all wrap objects from the body
    wrap_set: osim.WrapObjectSet = body.upd_WrapObjectSet()
    wrap_set.clearAndDestroy()

# Add new path points to the muscles
muscles : osim.SetMuscles = model.getMuscles()
for i in range(muscles.getSize()):
    muscle : osim.Muscle = muscles.get(i)
    muscle_name = muscle.getName()
    
    geo_path: osim.GeometryPath = muscle.getGeometryPath()
    path_points: osim.PathPointSet = geo_path.getPathPointSet()
    existing_path_points = path_points.getSize()
    
    path_wraps: osim.PathWrapSet = geo_path.getWrapSet()
    path_wraps.clearAndDestroy()
    # existing_wraps = path_wraps.getSize()
    # # Remove the old path wraps
    # for j in range(existing_wraps - 1, -1, -1):
    #     path_wraps.remove(j)
    
    # Add the new path points
    muscle_rows = young_attachments.filter(pl.col('Abbreviation') == muscle_name.split('R_')[-1])
    index = 1
    for row in muscle_rows.iter_rows(named=True):
        lower_frame = row['Frame'].lower()
        frame = lower_frame + '_r' if lower_frame != 'spine' else lower_frame
        mesh = model.getBodySet().get(frame)
        point_name = muscle_name + '_' + frame + '_' + row['Type'].lower() + '_' + str(index)
        young_loc = np.array([[row['X (mm)'], row['Y (mm)'], row['Z (mm)']]])/100 # Young has a weird scale
        loc = convert_points_between_meshes(young_loc, transforms[frame])
        vec = osim.Vec3(loc[0][0], loc[0][1], loc[0][2])
        muscle.addNewPathPoint(point_name, mesh, vec)
        logger.debug(f"Added path point for muscle: {muscle_name}, body: {mesh.getName()}, location: {loc}")
        index += 1
    # Remove the old path points
    for j in range(existing_path_points - 1, -1, -1):
        path_points.remove(j)
        logger.debug(f"Removed path point index: {j} for muscle: {muscle_name}")

model.finalizeFromProperties()
model.finalizeConnections()
out = model_dir / 'rat_hindlimb_millard_y2j.osim'
model.printToXML(out.as_posix())

2025-12-10 05:40:43.839 | DEBUG    | __main__:<module>:49 - Added path point for muscle: R_BFa, body: pelvis_r, location: [[-0.01217486  0.00924643 -0.00578144]]


2025-12-10 05:40:43.840 | DEBUG    | __main__:<module>:49 - Added path point for muscle: R_BFa, body: pelvis_r, location: [[-0.01228853  0.00508418 -0.00151152]]


2025-12-10 05:40:43.841 | DEBUG    | __main__:<module>:49 - Added path point for muscle: R_BFa, body: pelvis_r, location: [[-0.01233545  0.001285   -0.00012089]]


2025-12-10 05:40:43.842 | DEBUG    | __main__:<module>:49 - Added path point for muscle: R_BFa, body: tibia_r, location: [[-0.00297205  0.03250314  0.00135646]]


2025-12-10 05:40:43.842 | DEBUG    | __main__:<module>:54 - Removed path point index: 1 for muscle: R_BFa


...

2025-12-10 05:40:43.922 | DEBUG    | __main__:<module>:49 - Added path point for muscle: R_Pop, body: tibia_r, location: [[-0.00324222  0.03493462 -0.00047687]]


2025-12-10 05:40:43.922 | DEBUG    | __main__:<module>:49 - Added path point for muscle: R_Pop, body: tibia_r, location: [[ 0.0018743   0.02569243 -0.00219991]]


2025-12-10 05:40:43.923 | DEBUG    | __main__:<module>:54 - Removed path point index: 1 for muscle: R_Pop


2025-12-10 05:40:43.923 | DEBUG    | __main__:<module>:54 - Removed path point index: 0 for muscle: R_Pop


True

## Knee joint definition

** TODO: Show plots proving what we say **

In Johnson's orginal model, the tibia offset frame used as the parent frame for the knee joint is defined at the distal end of the tibia, and the spatial transform is handled by a `SimmSpline` component. When the model is scaled to subject-specific parameters, the knee motion is incorrect because the `SimmSpline` trajectory does not change, and the tibia offset frame does not move, so there is nothing to account for the new tibia length. To fix this, we adjust the tibia offset frame to be at the approximate joint center of the knee as determined by the instantaneous center of rotation (ICR) of the knee shown in @fig-knee

![Knee ICR](./figs/knee-icr.png){#fig-knee}

In [10]:
# TODO: Derive this programmatically
new_frame = [-0.00057, 0.0399598, 0.0038162]
rotation2 = [-0.087266499999999996851, -0.075049199999999996469, -0.064577200000000001268, -0.052359900000000000886, -0.040142600000000000504, -0.029670599999999998364, -0.017453300000000001452, -0.0052359900000000002621, 0.0052359900000000002621, 0.017453300000000001452, 0.029670599999999998364, 0.040142600000000000504, 0.052359900000000000886, 0.064577200000000001268]
rotation3 = [0.26179900000000000393, 0.26179900000000000393, 0.26179900000000000393, 0.26179900000000000393, 0.26179900000000000393, 0.26179900000000000393, 0.26179900000000000393, 0.26179900000000000393, 0.26179900000000000393, 0.26179900000000000393, 0.26179900000000000393, 0.26179900000000000393, 0.26179900000000000393, 0.26179900000000000393]
translation1 = [-0.0052385299999999999226, -0.0046464799999999997077, -0.0040425699999999996706, -0.0034725400000000000884, -0.0029796499999999999202, -0.0025907899999999999333, -0.0022994299999999998768, -0.0021100099999999998371, -0.001995639999999999914, -0.0019320699999999999246, -0.0018704800000000001026, -0.0017793500000000000722, -0.0016382700000000000474, -0.0014133999999999999862]
translation2 = [-0.034168400000000001548, -0.03425389999999999685, -0.034185599999999996546, -0.033972299999999996944, -0.033651100000000003232, -0.033278500000000002523, -0.032889799999999996816, -0.032550400000000000167, -0.032282499999999998697, -0.032116100000000001591, -0.032063200000000000034, -0.032110500000000000154, -0.032239700000000003077, -0.032414400000000002933]
translation3 = [0.0026030200000000001601, 0.0028034100000000001032, 0.0028536500000000001101, 0.002903750000000000081, 0.0028935800000000001971, 0.0027731800000000000894, 0.0027023500000000000645, 0.0026110500000000001763, 0.002469499999999999907, 0.0024171399999999999136, 0.0024041000000000001605, 0.0023809000000000000302, 0.002496410000000000122, 0.0026910400000000000119]

model = osim.Model(out.as_posix())
model.initSystem()
knee = osim.CustomJoint.safeDownCast(model.getJointSet().get('knee_r'))
tibia_offset = osim.PhysicalOffsetFrame.safeDownCast(knee.getChildFrame())
tibia_offset.set_translation(osim.Vec3(new_frame[0], new_frame[1], new_frame[2]))

spatial_transform = knee.get_SpatialTransform()
rot2 = spatial_transform.get_rotation2()
rot3 = spatial_transform.get_rotation3()
rot3.set_function(osim.Constant(rotation3[0]))
trans1 = spatial_transform.get_translation1()
simm1 = osim.SimmSpline.safeDownCast(trans1.get_function())
for i, value in enumerate(translation1):
    simm1.setY(i, value)
trans2 = spatial_transform.get_translation2()
simm2 = osim.SimmSpline.safeDownCast(trans2.get_function())
for i, value in enumerate(translation2):
    simm2.setY(i, value)
trans3 = spatial_transform.get_translation3()
simm3 = osim.SimmSpline.safeDownCast(trans3.get_function())
for i, value in enumerate(translation3):
    simm3.setY(i, value)

model.finalizeFromProperties()
model.finalizeConnections()
out = model_dir / 'rat_hindlimb_millard_y2j_knee_fixed.osim'
model.printToXML(out.as_posix())

[info] Loaded model RatHindlimbRight from file models/osim/rat_hindlimb_millard_y2j.osim


True

## Change marker set

Our model uses a slightly different marker set than the one used in Johnson's.

In [11]:
#TODO: Visualize
marker_set_path = model_dir / 'rat_hindlimb_unilateral_markerset.xml'
marker_set = osim.MarkerSet(marker_set_path.as_posix())
model = osim.Model(out.as_posix())
model.getMarkerSet().clearAndDestroy()
model.updateMarkerSet(marker_set)
out = model_dir / 'rat_hindlimb_millard_y2j_knee_fixed_markers.osim'
model.printToXML(out.as_posix())

[info] Loaded model RatHindlimbRight from file models/osim/rat_hindlimb_millard_y2j_knee_fixed.osim
[info] Updated markers in model RatHindlimbRight


True

## Update muscle parameters

The original Johnson model from 2008 had estimated parameters. In a subsequent paper in 2011, @tbl-johnson-params

In [12]:
#| label: tbl-johnson-params
parameter_dir = Path('./data/parameters')
johnson_params = pl.read_csv(parameter_dir / 'johnson_2011_parameters.csv')

md(tabulate(johnson_params, headers='keys', tablefmt='pipe'))

| 0               | 1               | 2               | 3                       | 4                        | 5              | 6                         | 7                        | 8                       | 9                      | 10                | 11                | 12              | 13             | 14              | 15                | 16                 | 17      | 18         | 19                    | 20                   | 21                 | 22                 | 23        | 24              | 25              | 26                     | 27                     | 28         | 29        | 30        | 31                | 32              | 33                       | 34                       | 35     | 36                  | 37                | 38                 | 39             | 40                 | 41               | 42              |
|:----------------|:----------------|:----------------|:------------------------|:-------------------------|:---------------|:--------------------------|:-------------------------|:------------------------|:-----------------------|:------------------|:------------------|:----------------|:---------------|:----------------|:------------------|:-------------------|:--------|:-----------|:----------------------|:---------------------|:-------------------|:-------------------|:----------|:----------------|:----------------|:-----------------------|:-----------------------|:-----------|:----------|:----------|:------------------|:----------------|:-------------------------|:-------------------------|:-------|:--------------------|:------------------|:-------------------|:---------------|:-------------------|:-----------------|:----------------|
| AB              | AL              | AM              | BFa                     | BFp                      | CF             | EDL                       | EHL                      | FDL                     | FHL                    | GI                | GS                | GMa             | GMe            | GMi             | GA                | GP                 | IP?     | IP         | LG                    | MG                   | OE                 | OI                 | Pec       | PerB            | Per             | PerQa                  | PerQi                  | Pir        | Pla       | Pop       | QF                | SM              | STa                      | STp                      | Sol    | TFL                 | TA                | TP                 | RF             | VI                 | VL               | VM              |
| adductor brevis | adductor longus | adductor magnus | biceps femoris anterior | biceps femoris posterior | caudofemoralis | extensor digitorum longus | extensor hallucis longus | flexor digitorum longus | flexor hallucis longus | gemellus inferior | gemellus superior | gluteus maximus | gluteus medius | gluteus minimus | gracilis anterior | gracilis posterior | iliacus | illiopsoas | lateral gastrocnemius | medial gastrocnemius | obturator externus | obturator internus | pectineus | peroneus brevis | peroneus longus | peroneus digiti quarti | peroneus digiti quinti | piriformis | plantaris | popliteus | quadratus femoris | semimembranosus | semitendinosus accessory | semitendinosus principal | soleus | tensor fascia latae | tibialis anterior | tibialis posterior | rectus femoris | vastus intermedius | vastus lateralis | vastus medialis |
| 1087            | 94              | 280             | 470                     | 1655                     | 275            | 146                       | 13                       | 470                     | 22                     | 55                | 12                | 793             | 1444           | 378             | 168               | 246                | 999     | 1179       | 805                   | 720                  | 194                | 54                 | 233       | 88              | 131             | 31                     | 21                     | 478        | 308       | 90        | 155               | 1291            | 177                      | 707                      | 155    | 362                 | 542               | 85                 | 900            | 181                | 1113             | 396             |
| 157             | 19              | 93              | 99                      | 348                      | 91             | 14                        | 2                        | 53                      | 1                      | 2                 | 2                 | 33              | 182            | 25              | 32                | 43                 | 77      | 126        | 161                   | 56                   | 13                 | 11                 | 17        | 11              | 16              | 4                      | 3                      | 60         | 27        | 5         | 14                | 320             | 25                       | 101                      | 27     | 15                  | 60                | 5                  | 67             | 12                 | 170              | 26              |
| 22.8            | 16.6            | 28.4            | 39.7                    | 37.6                     | 37.9           | 13.7                      | 12.1                     | 9.0                     | 10.8                   | 4.2               | 6.1               | 19.6            | 19.9           | 11.8            | 30.3              | 29.3               | 22.3    | 23.8       | 12.5                  | 14.6                 | 6.0                | 4.5                | 10.9      | 8.6             | 7.6             | 8.8                    | 8.6                    | 10.8       | 12.5      | 6.5       | 11.0              | 35.7            | 34.1                     | 43.6                     | 16.0   | 20.6                | 15.2              | 5.4                | 12.2           | 11.6               | 19.8             | 15.6            |
| 0.4             | 2.1             | 1.0             | 3.9                     | 1.7                      | 3.2            | 1.0                       | 0.8                      | 0.5                     | 1.0                    | 0.2               | 1.1               | 4.7             | 2.7            | 0.3             | 4.9               | 4.1                | 0.9     | 3.3        | 0.8                   | 0.4                  | 0.8                | 0.7                | 0.7       | 0.9             | 0.3             | 1.9                    | 0.8                    | 1.6        | 1.5       | 0.6       | 1.0               | 1.0             | 6.4                      | 10.4                     | 5.1    | 0.6                 | 0.7               | 1.4                | 0.9            | 0.6                | 0.3              | 1.3             |
| 12              | 9               | 0               | 0                       | 7                        | 0              | 10                        | 5                        | 11                      | 2                      | 18                | 10                | 12              | 12             | 16              | 0                 | 3                  | 7       | 15         | 9                     | 13                   | 16                 | 7                  | 7         | 6               | 9               | 7                      | 4                      | 16         | 11        | 13        | 3                 | 0               | 0                        | 0                        | 4      | 11                  | 9                 | 5                  | 11             | 14                 | 14               | 11              |
| 5               | 4               | 0               | 1                       | 5                        | 0              | 3                         | 3                        | 5                       | 4                      | 7                 | 14                | 10              | 2              | 9               | 0                 | 4                  | 4       | 3          | 6                     | 4                    | 6                  | 6                  | 1         | 7               | 2               | 5                      | 1                      | 8          | 4         | 6         | 2                 | 0               | 0                        | 0                        | 9      | 8                   | 4                 | 3                  | 5              | 3                  | 3                | 6               |
| 1246            | 106             | 308             | 346                     | 1274                     | 210            | 225                       | 25                       | 1124                    | 45                     | 291               | 38                | 942             | 1912           | 754             | 158               | 240                | 1043    | 1151       | 1863                  | 1188                 | 732                | 307                | 503       | 243             | 404             | 86                     | 56                     | 1137       | 572       | 298       | 306               | 1043            | 142                      | 463                      | 145    | 397                 | 878               | 361                | 1681           | 260                | 1517             | 620             |
| 80              | 15              | 24              | 23                      | 83                       | 25             | 8                         | 2                        | 217                     | 1                      | 38                | 13                | 276             | 233            | 61              | 10                | 35                 | 84      | 157        | 116                   | 92                   | 80                 | 35                 | 14        | 20              | 32              | 15                     | 3                      | 123        | 56        | 17        | 18                | 69              | 23                       | 148                      | 27     | 30                  | 81                | 83                 | 86             | 25                 | 79               | 73              |
| 12.21           | 1.039           | 3.02            | 3.39                    | 12.49                    | 2.06           | 2.21                      | 0.25                     | 11.02                   | 0.44                   | 2.85              | 0.37              | 9.23            | 18.74          | 7.39            | 1.55              | 2.35               | 10.22   | 11.28      | 18.26                 | 11.64                | 7.17               | 3.01               | 4.93      | 2.38            | 3.96            | 0.84                   | 0.55                   | 11.14      | 5.61      | 2.92      | 3.0               | 10.22           | 1.39                     | 4.54                     | 1.42   | 3.89                | 8.6               | 3.54               | 16.47          | 2.55               | 14.87            | 6.08            |
| 0.784           | 0.147           | 0.235           | 0.225                   | 0.813                    | 0.245          | 0.078                     | 0.02                     | 2.127                   | 0.01                   | 0.372             | 0.127             | 2.705           | 2.283          | 0.598           | 0.098             | 0.343              | 0.823   | 1.539      | 1.137                 | 0.902                | 0.784              | 0.343              | 0.137     | 0.196           | 0.314           | 0.147                  | 0.029                  | 1.205      | 0.549     | 0.167     | 0.176             | 0.676           | 0.225                    | 1.45                     | 0.265  | 0.294               | 0.794             | 0.813              | 0.843          | 0.245              | 0.774            | 0.715           |
|                 |                 |                 |                         |                          |                | 265                       |                          | 1344                    |                        |                   |                   |                 |                |                 |                   |                    |         |            | 1734                  | 1364                 |                    |                    |           | 501             | 622             | 139                    | 113                    |            | 792       |           |                   |                 |                          |                          | 234    |                     | 1004              | 605                | 1859           |                    |                  | 611             |
|                 |                 |                 |                         |                          |                | 65                        |                          | 494                     |                        |                   |                   |                 |                |                 |                   |                    |         |            | 43                    | 30                   |                    |                    |           | 40              | 122             | 2                      | 22                     |            | 13        |           |                   |                 |                          |                          | 29     |                     | 45                | 51                 | 71             |                    |                  | 87              |
| 406             | 93              | 506             | 706                     | 669                      | 675            | 243                       | 216                      | 160                     | 193                    | 74                | 109               | 349             | 355            | 211             | 539               | 163                | 402     | 418        | 223                   | 260                  | 106                | 81                 | 193       | 152             | 135             | 157                    | 152                    | 193        | 223       | 115       | 197               | 634             | 607                      | 775                      | 89     | 367                 | 271               | 97                 | 216            | 65                 | 352              | 278             |
| 21              | 17              | 32              | 60                      | 71                       | 77             | 15                        | 22                       | 33                      | 6                      | 9                 | 31                | 75              | 35             | 22              | 50                | 27                 | 50      | 64         | 17                    | 25                   | 10                 | 10                 | 2         | 16              | 12              | 36                     | 13                     | 19         | 17        | 5         | 15                | 57              | 95                       | 193                      | 22     | 27                  | 29                | 31                 | 19             | 9                  | 25               | 34              |
|                 |                 |                 |                         |                          |                | 205                       |                          | 140                     |                        |                   |                   |                 |                |                 |                   |                    |         |            | 209                   | 238                  |                    |                    |           | 117             | 113             | 116                    | 116                    |            | 207       |           |                   |                 |                          |                          | 93     |                     | 290               | 68                 | 227            |                    |                  | 161             |
|                 |                 |                 |                         |                          |                | 13                        |                          | 17                      |                        |                   |                   |                 |                |                 |                   |                    |         |            | 17                    | 6                    |                    |                    |           | 4               | 7               | 19                     | 4                      |            | 13        |           |                   |                 |                          |                          | 11     |                     | 6                 | 1                  | 13             |                    |                  | 3               |
| 0.0             | 4.5             | 0.0             | 0.0                     | 0.0                      | 0.0            | 9.0                       | 13.6                     | 8.7                     | 9.7                    | 0.0               | 0.0               | 4.8             | 0.0            | 0.0             | 0.0               | 0.9                |         | 4.1        | 9.4                   | 9.1                  | 0.0                | 2.8                | 0.0       | 10.3            | 12.7            | 9.1                    | 19.6                   | 0.0        | 6.7       | 0.0       | 0.0               | 0.0             | 0.0                      | 0.0                      | 9.5    | 0.0                 | 11.9              | 12.4               | 6.6            | 6.6                | 6.6              | 6.6             |
| 0               | 0.3             | 0               | 0                       | 0                        | 0              | 0.4                       | 1.2                      | 0.8                     | 0.0                    | 0                 | 0                 | 0.4             | 0              | 0               | 0                 | 0.1                |         | 0.0        | 0.3                   | 0.3                  | 0                  | 4.0                | 0         | 0.2             | 0.1             | 1.4                    | 0.9                    | 0          | 0.2       | 0         | 0                 | 0               | 0                        | 0                        | 0.5    | 0                   | 1.5               | 0.2                | 0.8            | 0.8                | 0.8              | 0.8             |

In [13]:
#| label: tbl-eng-params
eng_params = pl.read_csv(parameter_dir / 'eng_2008_parameters.csv')

md(tabulate(eng_params, headers='keys', tablefmt='pipe'))

| 0               | 1               | 2               | 3                | 4              | 5            | 6                         | 7                        | 8                       | 9                      | 10               | 11               | 12             | 13                    | 14                   | 15                    | 16            | 17            | 18            | 19          | 20             | 21           | 22       | 23              | 24              | 25              | 26           | 27      | 28             | 29              | 30             | 31        | 32                | 33                 | 34                 | 35               | 36             |
|:----------------|:----------------|:----------------|:-----------------|:---------------|:-------------|:--------------------------|:-------------------------|:------------------------|:-----------------------|:-----------------|:-----------------|:---------------|:----------------------|:---------------------|:----------------------|:--------------|:--------------|:--------------|:------------|:---------------|:-------------|:---------|:----------------|:----------------|:----------------|:-------------|:--------|:---------------|:----------------|:---------------|:----------|:------------------|:-------------------|:-------------------|:-----------------|:---------------|
| AB              | AL              | AM              |                  | BF             |              | EDL                       | EHL                      | FDL                     | FHL                    | LG               | MG               | GMe            | CF                    | GA                   | GP                    |               |               |               |             |                |              | MX       | PerB            | PerL            |                 | Pla          | IP      | RF             | SM              | ST             | Sol       | TA                | TP                 | VI                 | VL               | VM             |
| Adductor brevis | Adductor longus | Adductor magnus | Adductor tertius | Biceps femoris | Dorsiflexors | Extensor digitorum longus | Extensor hallucis longus | Flexor digitorum longus | Flexor hallucis longis | Gastrocnemius LH | Gastrocnemius MH | Gluteus medius | Gluteus superficialis | Gracilis caudal head | Gracilis cranial head | Hip abductors | Hip adductors | Hip extensors | Hip flexors | Knee extensors | Knee flexors | Muscle X | Peroneus brevis | Peroneus longus | Plantarflexors† | Plantaris*,† | Psoas   | Rectus femoris | Semimembranosus | Semitendinosus | Soleus*,† | Tibialis anterior | Tibialis posterior | Vastus intermedius | Vastus lateralis | Vastus medalis |
| 66.83           | 235.5           | 649.83          | 685.17           | 2670.83        | 271.17       | 131.83                    | 19.5                     | 463.83                  | 29.67                  | 1031.17          | 849.17           | 1992.33        | 521.0                 | 276.0                | 173.0                 | 1256.67       | 402.3         | 1300.64       | 1365.08     | 711.83         | 890.81       | 377.0    | 124.67          | 177.5           | 371.56          | 397.5        | 1784.83 | 945.33         | 1452.83         | 789.83         | 134.67    | 662.17            | 135.83             | 170.67             | 1288.83          | 442.5          |
| 3.26            | 11.86           | 16.49           | 18.37            | 95.48          | 15.9         | 3.24                      | 0.67                     | 10.87                   | 1.69                   | 30.53            | 24.18            | 47.38          | 31.66                 | 38.68                | 27.68                 | 735.67        | 118.84        | 369.73        | 419.75      | 250.48         | 261.32       | 17.21    | 4.45            | 7.01            | 117.83          | 16.87        | 296.1   | 33.23          | 39.84           | 33.17          | 4.57      | 17.7              | 9.71               | 15.53              | 41.22            | 44.81          |
| 1.16            | 1.07            | 2.21            | 1.98             | 3.4            | 1.39         | 1.34                      | 1.18                     | 0.97                    | 0.89                   | 1.56             | 1.61             | 2.43           | 1.92                  | 2.64                 | 2.57                  | 2.18          | 1.28          | 3.19          | 1.76        | 1.49           | 2.72         | 3.5      | 0.82            | 0.89            | 1.19            | 1.37         | 2.36    | 1.16           | 3.09            | 4.77           | 1.97      | 1.64              | 0.6                | 1.14               | 1.96             | 1.7            |
| 0.07            | 0.08            | 0.08            | 0.04             | 0.06           | 0.13         | 0.1                       | 0.07                     | 0.06                    | 0.03                   | 0.14             | 0.13             | 0.11           | 0.11                  | 0.08                 | 0.13                  | 0.25          | 0.39          | 0.4           | 0.6         | 0.2            | 0.37         | 0.08     | 0.02            | 0.08            | 0.15            | 0.07         | 0.23    | 0.1            | 0.13            | 0.05           | 0.19      | 0.07              | 0.04               | 0.08               | 0.12             | 0.04           |
| 0.0             | 4.2             | 0.0             | 0.6              | 3.6            | 9.17         | 9.0                       | 5.7                      | 19.3                    | 7.5                    | 14.2             | 14.0             | 0.6            | -1.1                  | 0.0                  | 0.0                   | -0.28         | 0.94          | 1.02          | 21.25       | 16.77          | 5.69         | 1.0      | 12.6            | 20.0            | 14.88           | 16.4         | 17.1    | 25.4           | 2.1             | 0.0            | 3.9       | 12.8              | 26.0               | 6.9                | 10.0             | 24.7           |
| 0.0             | 1.7             | 0.0             | 0.6              | 3.5            | 2.05         | 1.1                       | 1.6                      | 1.9                     | 2.1                    | 3.6              | 3.9              | 0.8            | 0.7                   | 0.0                  | 0.0                   | 0.83          | 0.81          | 0.67          | 4.17        | 4.83           | 2.33         | 1.0      | 2.8             | 3.1             | 2.21            | 3.2          | 0.8     | 4.0            | 2.0             | 0.0            | 2.4       | 1.2               | 7.3                | 1.2                | 3.3              | 3.4            |
| 1.57            | 1.62            | 2.98            | 2.47             | 5.1            | 2.49         | 2.81                      | 1.74                     | 3.22                    | 1.81                   | 3.42             | 3.61             | 3.64           | 2.55                  | 3.47                 | 3.04                  | 3.09          | 2.42          | 4.3           | 4.38        | 3.01           | 4.14         | 4.31     | 2.92            | 2.76            | 3.04            | 4.08         | 5.55    | 3.21           | 4.27            | 5.95           | 2.83      | 2.91              | 2.67               | 2.64               | 3.42             | 2.79           |
| 0.09            | 0.06            | 0.09            | 0.07             | 0.08           | 0.38         | 0.09                      | 0.09                     | 0.05                    | 0.19                   | 0.08             | 0.06             | 0.16           | 0.17                  | 0.14                 | 0.15                  | 0.54          | 0.37          | 0.48          | 1.17        | 0.18           | 0.31         | 0.12     | 0.11            | 0.04            | 0.22            | 0.07         | 0.26    | 0.11           | 0.12            | 0.09           | 0.19      | 0.04              | 0.01               | 0.16               | 0.08             | 0.09           |
| 0.06            | 0.21            | 0.28            | 0.33             | 0.74           | 0.16         | 0.09                      | 0.02                     | 0.44                    | 0.03                   | 0.62             | 0.49             | 0.79           | 0.26                  | 0.1                  | 0.06                  | 0.52          | 0.2           | 0.41          | 0.69        | 0.42           | 0.33         | 0.1      | 0.14            | 0.19            | 0.27            | 0.26         | 0.67    | 0.71           | 0.45            | 0.16           | 0.07      | 0.38              | 0.19               | 0.14               | 0.62             | 0.22           |
| 0.0             | 0.01            | 0.01            | 0.01             | 0.03           | 0.11         | 0.01                      | 0.0                      | 0.03                    | 0.0                    | 0.05             | 0.04             | 0.05           | 0.01                  | 0.02                 | 0.01                  | 0.26          | 0.05          | 0.1           | 0.02        | 0.14           | 0.08         | 0.01     | 0.01            | 0.02            | 0.07            | 0.01         | 0.07    | 0.05           | 0.02            | 0.02           | 0.01      | 0.02              | 0.03               | 0.01               | 0.05             | 0.02           |
| 0.75            | 0.66            | 0.74            | 0.8              | 0.67           | 0.58         | 0.48                      | 0.68                     | 0.3                     | 0.51                   | 0.46             | 0.45             | 0.67           | 0.56                  | 0.76                 | 0.84                  | 0.72          | 0.75          | 0.74          | 0.39        | 0.5            | 0.65         | 0.81     | 0.28            | 0.32            | 0.4             | 0.34         | 0.42    | 0.36           | 0.72            | 0.8            | 0.69      | 0.57              | 0.23               | 0.44               | 0.57             | 0.61           |
| 0.02            | 0.04            | 0.02            | 0.02             | 0.02           | 0.06         | 0.05                      | 0.02                     | 0.02                    | 0.04                   | 0.05             | 0.04             | 0.05           | 0.03                  | 0.02                 | 0.01                  | 0.04          | 0.03          | 0.03          | 0.03        | 0.06           | 0.06         | 0.01     | 0.01            | 0.03            | 0.05            | 0.02         | 0.02    | 0.04           | 0.02            | 0.01           | 0.04      | 0.03              | 0.01               | 0.04               | 0.04             | 0.02           |

Thus, Johnson's parameters were chosen for the model.

In [14]:
model = osim.Model(out.as_posix())
muscles : osim.SetMuscles = model.getMuscles()
for i in range(muscles.getSize()):
    muscle : osim.Muscle = muscles.get(i)
    muscle_name = muscle.getName().replace('R_', '')
    params = johnson_params.row(by_predicate=pl.col('Abbreviation') == muscle_name, named=True)
    if params is None:
        print(f"Muscle {muscle_name} not found in parameters file.")
        continue
    f0 = params['Fo (N)']
    l0 = params['l0 (mm)'] / 1000  # Convert mm to m
    lts = params['lts (mm)'] / 1000  # Convert mm to m
    alpha = params['θ0 (deg)'] * np.pi / 180  # Convert deg to rad
    muscle.setMaxIsometricForce(f0)
    muscle.setOptimalFiberLength(l0)
    muscle.setTendonSlackLength(lts)
    muscle.setPennationAngleAtOptimalFiberLength(alpha)
    print(f"Updated {muscle_name}: F0={f0}, l0={l0}, lts={lts}, alpha={alpha}")

model.finalizeFromProperties()
model.finalizeConnections()
out = model_dir / 'rat_hindlimb_millard_y2j_knee_fixed_markers_params.osim'
model.printToXML(out.as_posix())

[info] Loaded model RatHindlimbRight from file models/osim/rat_hindlimb_millard_y2j_knee_fixed_markers.osim
Updated BFa: F0=3.39, l0=0.039700000000000006, lts=0.0, alpha=0.0
Updated BFp: F0=12.49, l0=0.0376, lts=0.0, alpha=0.12217304763960307
Updated CF: F0=2.06, l0=0.037899999999999996, lts=0.0, alpha=0.0
Updated STp: F0=4.54, l0=0.0436, lts=0.0, alpha=0.0
Updated STa: F0=1.39, l0=0.0341, lts=0.0, alpha=0.0
Updated SM: F0=10.22, l0=0.0357, lts=0.0, alpha=0.0
Updated QF: F0=3.0, l0=0.011, lts=0.0, alpha=0.05235987755982988
Updated TFL: F0=3.89, l0=0.0206, lts=0.0, alpha=0.19198621771937624
Updated GMa: F0=9.23, l0=0.019600000000000003, lts=0.0048, alpha=0.20943951023931953
Updated GMe: F0=18.74, l0=0.019899999999999998, lts=0.0, alpha=0.20943951023931953
Updated GMi: F0=7.39, l0=0.011800000000000001, lts=0.0, alpha=0.2792526803190927
Updated Pir: F0=11.14, l0=0.0108, lts=0.0, alpha=0.2792526803190927
Updated GP: F0=2.35, l0=0.0293, lts=0.0009, alpha=0.05235987755982988
Updated GA: F0=1

True

## Update mass and inertia properties


## Estimate tendon slack length

Johnson reports tendon slack lengths based on anatomical measurements. However, the literature reports large inaccuracies for tendon slack length values measured this way. It is also said to be one of the most influential parameters in simulating muscle force production. Its role, though based in a physiological concept, is primarily a model parameter that helps define the force-length behavior of the muscle-tendon unit. 

A method for reducing the error in tendon slack length estimation is to optimize it based on muscle fiber lengths observed during motion. This method was originally proposed in [Manal 2008] but was limited to a single muscle crossing a single joint. Subsequent works, namely Charles 2016, have applied this methodology to an array of muscles crossing multiple joints. 

The method can be summarized as follows:

Two ranges of motion are used to estimate the optimal fiber length and tendon slack length of muscles based on observed fiber lengths during motion. The first is the model's full range of motion, and the second is the observed range of motion during walking.

In [15]:
from movedb.osim import OsimGraph


## Mirror the model

To mirror the right side of the model to the left, a series of steps needs to happen:
- Duplicate bodies
    - Follow the naming convention of `{body}_l` and `{body}_r`
    - For the left side, the attached_geometry will be the same as the right side, but the scale_factors will be 1 1 -1
    - The mass and inertia properties will need to be updated to reflect the new bodies
- Duplicate joints 
    - Follow the naming convention of `{joint}_l` and `{joint}_r`
    - Frames that reference bodies will need to be updated to reference the new bodies
    - Physical offset frames with Z translations will need to be negated
    - Spatial transforms with Z translations will need to be negated
- Duplicate muscles
    - Follow the naming convention of `L_{muscle}` and `R_{muscle}`
    - Path points will need to be updated to reference the new bodies
    - For the left side Z translations will need to be negated
- Duplicate markers 
    - Follow the naming convention of `L{marker}` and `R{marker}`
    - Markers will need to be updated to reference the new bodies
    - For the left side Z translations will need to be negated

In [16]:
model = osim.Model(out.as_posix())
# TODO: Turn this into functions
body_set: osim.BodySet = model.getBodySet()
cloned_bodies = {}
n_bodies = body_set.getSize()
for i in range(n_bodies):
    body: osim.Body = model.getBodySet().get(i)
    body_name: str = body.getName()
    new_body_name = body_name
    if body_name != 'spine': # Spine does not get mirrored
        new_body: osim.Body = body.clone()
        new_body_name = body_name.replace('_r', '_l')
        new_body.setName(new_body_name)
        geom: osim.Geometry = new_body.get_attached_geometry(0)
        geom.setName(geom.getName().replace('_r', '_l'))
        geom.set_scale_factors(osim.Vec3(1, 1, -1))
        com: osim.Vec3 = new_body.getMassCenter()
        new_body.setMassCenter(osim.Vec3([com.get(0), com.get(1), -com.get(2)]))
        moi: osim.Inertia = new_body.getInertia()
        moments: osim.Vec3 = moi.getMoments()
        products: osim.Vec3 = moi.getProducts()
        new_body.setInertia(osim.Inertia(
                            moments.get(0), moments.get(1), moments.get(2),
                            products.get(0), -products.get(1), -products.get(2)))
        model.addBody(new_body)
    cloned_bodies[body_name] = new_body_name

joint_set: osim.JointSet = model.getJointSet()
left_joint_set: osim.JointSet = joint_set.clone()
# Remove ground_spine
left_joint_set.remove(left_joint_set.get('ground_spine'))

left_joint_set_file = './models/osim/left_joint_set.xml'
left_joint_set.printToXML(left_joint_set_file)
with open(left_joint_set_file, 'r') as file:
    joint_set_content = file.read()

for old_body_name, new_body_name in cloned_bodies.items():
    joint_set_content = joint_set_content.replace(old_body_name, new_body_name)

new_joint_names = []
for i in range(left_joint_set.getSize()):
    joint: osim.Joint = left_joint_set.get(i)
    joint_name: str = joint.getName()
    new_joint_name: str = joint_name.replace('_r', '_l')
    joint_set_content = joint_set_content.replace(joint_name, new_joint_name)
    new_joint_names.append(new_joint_name)
    
with open(left_joint_set_file, 'w') as file:
    file.write(joint_set_content)
    
left_joint_set = osim.JointSet(left_joint_set_file)
# Remove the left_joint_set file
import os
os.remove(left_joint_set_file)
    
js: osim.JointSet = model.getJointSet()
for i in range(left_joint_set.getSize()):
    js.cloneAndAppend(left_joint_set.get(i))

model.initSystem()
for joint_name in new_joint_names:
    joint: osim.CustomJoint = osim.CustomJoint.safeDownCast(model.getJointSet().get(joint_name))
    if joint is None:
        print(f"Joint {joint_name} not found in the model.")
        continue
    # Mirror the parent and child frames
    parent_offset: osim.PhysicalOffsetFrame = osim.PhysicalOffsetFrame.safeDownCast(joint.getParentFrame())
    parent_pos = parent_offset.get_translation()
    parent_rot = parent_offset.get_orientation()
    child_offset: osim.PhysicalOffsetFrame = osim.PhysicalOffsetFrame.safeDownCast(joint.getChildFrame())
    child_pos = child_offset.get_translation()
    child_rot = child_offset.get_orientation()
    parent_offset.set_translation(osim.Vec3(parent_pos.get(0), parent_pos.get(1), -parent_pos.get(2)))
    child_offset.set_translation(osim.Vec3(child_pos.get(0), child_pos.get(1), -child_pos.get(2)))
    
    # Mirror the spatial transform
    spatial_transform: osim.SpatialTransform = joint.get_SpatialTransform()
    transform_axes: list[osim.Vec3] = spatial_transform.getAxes()
    for i, vec in enumerate(transform_axes):
        # For rotations, negate x and y components
        # For translations, negate z component
        if i <= 2 and vec[2]:  # Skip the first three axes (rotation1-3)
            continue
        elif i > 2 and not vec[2]:  # Skip the first three axes (translation1-3)
            continue
            
        # Assume that order is always rotation1-3, translation1-3
        get_transform_function = ('get_rotation' + str(i+1)) if i < 3 else ('get_translation' + str(i-2))
        transform_axis: osim.TransformAxis = getattr(spatial_transform, get_transform_function)()
        axis_function: osim.Function = transform_axis.getFunction()
        concreteClass = axis_function.getConcreteClassName()
        match concreteClass:
            case 'SimmSpline':
                # Mirror the function
                simm_spline: osim.SimmSpline = osim.SimmSpline.safeDownCast(axis_function)
                if simm_spline is None:
                    raise ValueError("Function is not a SimmSpline.")
                for k in range(simm_spline.getSize()):
                    y: float = simm_spline.getY(k)
                    simm_spline.setY(k, -y)
            case 'LinearFunction':
                linear_func: osim.LinearFunction = osim.LinearFunction.safeDownCast(axis_function)
                if linear_func is None:
                    raise ValueError("Function is not a LinearFunction.")
                linear_func.setSlope(-linear_func.getSlope())
                linear_func.setIntercept(-linear_func.getIntercept())
            case 'Constant':
                constant_func: osim.Constant = osim.Constant.safeDownCast(axis_function)
                if constant_func is None:
                    raise ValueError("Function is not a Constant.")
                constant_func.setValue(-constant_func.getValue())
            case 'MultiplierFunction':
                mult_func: osim.MultiplierFunction = osim.MultiplierFunction.safeDownCast(axis_function)
                if mult_func is None:
                    raise ValueError("Function is not a MultiplierFunction.")
                # For MultiplierFunction, negate the scale
                mult_func.setScale(-mult_func.getScale())
            case _:
                print(f"Unsupported function type: {concreteClass}, not mirroring")

# Update muscles 
muscle_set: osim.ForceSet = model.getForceSet()
left_muscle_set_file = 'models/osim/left_muscle_set.xml'
muscle_set.printToXML(left_muscle_set_file)
with open(left_muscle_set_file, 'r') as file:
    muscle_set_content = file.read()

for old_body_name, new_body_name in cloned_bodies.items():
    muscle_set_content = muscle_set_content.replace(old_body_name, new_body_name)
    
new_muscle_names = []
for i in range(muscle_set.getSize()):
    muscle: osim.Muscle = muscle_set.get(i)
    muscle_name: str = muscle.getName()
    new_muscle_name: str = muscle_name.replace('R_', 'L_')
    muscle_set_content = muscle_set_content.replace(muscle_name, new_muscle_name)
    new_muscle_names.append(new_muscle_name)

for old_body_name, new_body_name in cloned_bodies.items():
    muscle_set_content = muscle_set_content.replace(old_body_name, new_body_name)

with open(left_muscle_set_file, 'w') as file:
    file.write(muscle_set_content)
    
left_muscle_set = osim.ForceSet(left_muscle_set_file)
os.remove(left_muscle_set_file)

for muscle_name in new_muscle_names:
    muscle: osim.Muscle = osim.Muscle.safeDownCast(left_muscle_set.get(muscle_name))
    path_points: osim.PathPointSet = muscle.getGeometryPath().getPathPointSet()
    for i in range(path_points.getSize()):
        path_point: osim.PathPoint = osim.PathPoint.safeDownCast(path_points.get(i))
        loc: osim.Vec3 = path_point.get_location()
        path_point.setLocation(osim.Vec3(loc.get(0), loc.get(1), -loc.get(2)))
    muscle_set.append(muscle)

# Make sure to update the model before printing
model.finalizeFromProperties()
model.finalizeConnections()

# For some reason this has to happen after finalizing connections
marker_set = osim.MarkerSet('models/osim/rat_hindlimb_bilateral_markerset.xml')
model.getMarkerSet().clearAndDestroy()
model.updateMarkerSet(marker_set)

model.printToXML('models/osim/rat_hindlimb_bilateral.osim')

[info] Loaded model RatHindlimbRight from file models/osim/rat_hindlimb_millard_y2j_knee_fixed_markers_params.osim


[info] Updated markers in model RatHindlimbRight


True

## Conclusion
TODO: Summarize the changes made to the model and potential future work.